> VIGOR 2.12.1 example. Install with `pip install "VIGOR[io,progress,plot] @ git+https://github.com/sanchestm/VIGOR"` (see the [README](../../README.md)). Outputs are cleared: this notebook was checked against the 2.12.1 API but not re-executed for this release.

# Real26: VIGOR, genuine GDEC 3.1.3 and rrBLUP

Individually migrated from `_usethis_blup_comparison_domino_latest_chrgrm_experts(2).ipynb`. See `MIGRATION.md` for original-cell lineage and deferred stages.

The default main path fits all **26 actual traits and 7,616 animals**, with deterministic whole-animal and within-animal holdouts. It uses the pinned **r11.2.1** transductive GRM; a new r11.3 cohort needs its own paired fixture and cache. Trait scaling and selection use observed training data only. Previously inspected split and upstream pre-regression limit scientific inference.

Set `VIGOR_RUN_EXTENDED=1` for bounded real26 factorGWAS, fine mapping and covariance exploration. This does not run a whole genome/server mega analysis. Optional stages are reported separately; disabled branches are skipped, never counted as scientific tests. No Optuna search runs by default.

In [ ]:
from pathlib import Path
import os, sys
# Optional server overrides: VIGOR_SOURCE, VIGOR_REAL26_FIXTURE, VIGOR_PLINK_PREFIX,
# VIGOR_GDEC, VIGOR_RSCRIPT, VIGOR_OUTPUT, VIGOR_THREADS (1..20).
SOURCE = Path(os.environ.get('VIGOR_SOURCE', Path.cwd().parents[1])).resolve()
if not (SOURCE/'vigor'/'__init__.py').is_file():
    candidates = [Path.cwd(), *Path.cwd().parents]
    SOURCE = next(p for p in candidates if (p/'vigor'/'__init__.py').is_file())
sys.path.insert(0, str(SOURCE))
for name in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS'):
    os.environ[name] = os.environ.get('VIGOR_THREADS', '2')
import numpy as np, pandas as pd
get_ipython().run_line_magic('matplotlib', 'inline')
import matplotlib.pyplot as plt
import vigor
from examples.real26_workflow import (configuration, load_fixture, run_prediction,
    run_comparators, run_extended, write_json)
assert Path(vigor.__file__).resolve() == SOURCE/'vigor'/'__init__.py'
print({'version':vigor.__version__, 'source':'selected candidate', 'threads':os.environ['OMP_NUM_THREADS']})


In [ ]:
config = configuration('prediction')
data = load_fixture(config)
print({'shape':data['ztrain'].shape, 'heldout_cells':{k:int(v.sum()) for k,v in data['masks'].items()}, 'assembly':'r11.2.1'})

## Current production covariance selection and prediction
Sparse G/E rank-grid CV uses only masked training traits. Inspect failed ranks; a successful full-rank fit is not proof that every candidate converged. Inference G/E and prediction profiles serve different purposes. The inherited residual-retention scalar has project-specific provenance; no claim of universal optimality is made.

In [ ]:
prediction = run_prediction(data, config)
fit = prediction['covariance_fit']
selection = prediction['prediction_sparse_cv']
display(pd.DataFrame(selection['candidates'])[['state','config']])
metrics = pd.read_csv(config['output']/'vigor_metrics.csv')
display(metrics.groupby(['target','split']).predictive_r2.mean().rename('macro_predictive_R2').to_frame())

## Genuine independently fitted comparators and shared-covariance parity
GDEC source is pinned by hash and checked for 3.1.3 identity. rrBLUP runs native R REML, one trait at a time, on the identical ordered kernel/training input. Shared-G/E parity checks the prediction solver; independently fitted metrics compare different estimators. Availability and actual failures are explicit.

In [ ]:
comparators = run_comparators(data, config, prediction)
all_metrics = pd.concat([metrics,comparators],ignore_index=True)
display(all_metrics.groupby(['model','target','split']).predictive_r2.mean().to_frame())

In [ ]:
panel = all_metrics[all_metrics.target.eq('genetic_mean')]
plot = panel.groupby(['model','split']).predictive_r2.mean().unstack('split')
ax = plot.plot.bar(figsize=(9,4),ylabel='Mean trait predictive R²',rot=15)
ax.set_title('Frozen real26 phenotypes; one inspected split')
ax.figure.tight_layout(); ax.figure.savefig(config['output']/'heldout_comparison.png',dpi=140)
plt.show()

## Explicit bounded optional scientific stages
With the toggle enabled, use all 26 traits on 512 selected animals and 128 real markers in two windows. Marker selection is phenotype-independent; factorGWAS refits LOCO backgrounds on this panel. Fine mapping uses 24 markers and observed training outcomes for one predeclared-by-training-coverage trait. This is an execution example, without genome-wide calibration, significance or causal-identification claims.

In [ ]:
extended = run_extended(data, config, prediction)
print(extended)

## Optional large analyses and scientific limitations
Full-server genome scans, graph/haplotype fine mapping, SHAP, annotation/ontology enrichment, additional A/D/GxE kernels, bootstrap coverage and Optuna are not enabled by this bounded run. Ontology evaluation requires supplied trait annotations; no labels are invented. Plotting/native-factor and holdout-comparison intent are retained above; historical detailed exploratory cell intent and locations are in MIGRATION.md. Fixed-support analytic SE excludes selection uncertainty; any future SE run must use the exact original phenotype digest and record probes. These stages remain **unrun**, not validated by executing their disabled branches.

In [ ]:
write_json(config['output']/'notebook_summary.json', {'state':'COMPLETE','shape':list(data['ztrain'].shape),'extended_state':extended['state'],'assembly':'r11.2.1','unrun':['whole-genome/server scale','ontology without annotations','SHAP','new A/D/GxE kernels','analytic SE selection uncertainty']})
print('Completed real26 workflow; this remains an unapproved candidate.')